<a href="https://colab.research.google.com/github/2873991-Manthena/Dissertation/blob/main/Traditional_Models_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Import Required Libraries

## Objective

This notebook implements baseline traditional machine learning models for MBTI personality prediction using the preprocessed social media datasets.

The implementation includes text vectorization using TF-IDF followed by multiple supervised learning algorithms. The performance of each model will be evaluated using standard classification metrics including accuracy, precision, recall, and F1-score.

In [ ]:
###############################################################
# IMPORT LIBRARIES
###############################################################

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression

from sklearn.naive_bayes import MultinomialNB

from sklearn.svm import LinearSVC

from sklearn.ensemble import RandomForestClassifier

print("All libraries imported successfully!")

All libraries imported successfully!


# 2. Load Preprocessed Dataset

## Objective

The preprocessed MBTI datasets generated during the preprocessing stage are loaded for model development. These datasets contain the cleaned textual content and corresponding MBTI personality labels.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
###############################################################
# LOAD DATASETS FROM GOOGLE DRIVE
###############################################################

import pandas as pd

DATASET_PATH = "/content/drive/MyDrive/Dissertation/Datasets/"

mbti1 = pd.read_csv(DATASET_PATH + "mbti1_preprocessed.csv")
mbti500 = pd.read_csv(DATASET_PATH + "mbti500_preprocessed.csv")

print("✅ Datasets loaded successfully!")

print(f"MBTI1 Shape   : {mbti1.shape}")
print(f"MBTI500 Shape : {mbti500.shape}")

✅ Datasets loaded successfully!
MBTI1 Shape   : (8675, 2)
MBTI500 Shape : (106067, 2)


In [ ]:
###############################################################
# PREVIEW DATASETS
###############################################################

display(mbti1.head())

display(mbti500.head())

,type,clean_text
0,INFJ,intj moment sportscenter top ten play prank li...
1,ENTP,I find lack I post alarm sex boring position o...
2,INTP,good one course I say I know my blessing my cu...
3,INTJ,dear intp I enjoy our conversation day esoteri...
4,ENTJ,you fire another silly misconception approach ...


,type,clean_text
0,INTJ,know intj tool use interaction people excuse a...
1,INTJ,rap music ehh opp yeah know valid well know fa...
2,INTJ,preferably p hd low except wew lad video p min...
3,INTJ,drink like wish could drink red wine give head...
4,INTJ,space program ah bad deal meing freelance max ...


# 3. Select Dataset for Initial Model Development

## Objective

To establish baseline machine learning performance, the MBTI Dataset 1 is selected for the initial experiments. This dataset contains balanced textual samples and requires significantly less computational time than MBTI-500, making it suitable for baseline model development.

In [ ]:
###############################################################
# SELECT DATASET
###############################################################

df = mbti1.copy()

# Remove missing values
df = df.dropna(subset=["clean_text"])

# Remove empty strings
df = df[df["clean_text"].str.strip() != ""]

# Convert text to string
df["clean_text"] = df["clean_text"].astype(str)

# Reset index
df = df.reset_index(drop=True)

print("Working Dataset Shape :", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

Working Dataset Shape : (8674, 2)

Missing Values:
type          0
clean_text    0
dtype: int64


# 4. Encode Personality Labels

## Objective

Machine learning algorithms require numerical target labels. Therefore, the sixteen MBTI personality types are converted into numerical class labels using Label Encoding while preserving the mapping between each personality type and its encoded representation.

In [ ]:
###############################################################
# LABEL ENCODING
###############################################################

encoder = LabelEncoder()

df["label"] = encoder.fit_transform(df["type"])

print("Encoding Completed!")

mapping = pd.DataFrame({

    "Personality": encoder.classes_,

    "Encoded Label": range(len(encoder.classes_))

})

display(mapping)

Encoding Completed!


,Personality,Encoded Label
0,ENFJ,0
1,ENFP,1
2,ENTJ,2
3,ENTP,3
4,ESFJ,4
5,ESFP,5
6,ESTJ,6
7,ESTP,7
8,INFJ,8
9,INFP,9


# 5. Train-Test Split

## Objective

The dataset is divided into training and testing subsets using an 80:20 ratio. Stratified sampling is employed to preserve the distribution of MBTI personality classes in both subsets, ensuring a fair evaluation of model performance.

In [ ]:
###############################################################
# FEATURES AND LABELS
###############################################################

X = df["clean_text"]

y = df["label"]

In [ ]:
###############################################################
# TRAIN TEST SPLIT
###############################################################

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

print("Training Samples :", len(X_train))

print("Testing Samples :", len(X_test))

Training Samples : 6939
Testing Samples : 1735


# 6. TF-IDF Feature Extraction

## Objective

Term Frequency-Inverse Document Frequency (TF-IDF) transforms textual data into numerical feature vectors by assigning greater importance to informative words while reducing the influence of frequently occurring terms. These vectors serve as input features for traditional machine learning classifiers.


In [ ]:
###############################################################
# TF-IDF VECTORIZATION
###############################################################

tfidf = TfidfVectorizer(

    max_features=10000,

    ngram_range=(1,2),

    min_df=3,

    max_df=0.90

)

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF Feature Extraction Completed!")

print("Training Matrix :", X_train_tfidf.shape)

print("Testing Matrix :", X_test_tfidf.shape)

TF-IDF Feature Extraction Completed!
Training Matrix : (6939, 10000)
Testing Matrix : (1735, 10000)


TF-IDF Feature Extraction Completed!
Training Matrix : (6939, 10000)
Testing Matrix : (1735, 10000)

In [ ]:
###############################################################
# LOGISTIC REGRESSION
###############################################################

lr = LogisticRegression(

    max_iter=1000,

    random_state=42,

    n_jobs=-1

)

lr.fit(X_train_tfidf, y_train)

lr_predictions = lr.predict(X_test_tfidf)

print("Logistic Regression Training Completed!")

Logistic Regression Training Completed!


In [ ]:
###############################################################
# LOGISTIC REGRESSION RESULTS
###############################################################

lr_accuracy = accuracy_score(y_test, lr_predictions)

lr_precision = precision_score(

    y_test,

    lr_predictions,

    average="weighted"

)

lr_recall = recall_score(

    y_test,

    lr_predictions,

    average="weighted"

)

lr_f1 = f1_score(

    y_test,

    lr_predictions,

    average="weighted"

)

print(f"Accuracy : {lr_accuracy:.4f}")

print(f"Precision : {lr_precision:.4f}")

print(f"Recall : {lr_recall:.4f}")

print(f"F1 Score : {lr_f1:.4f}")

Accuracy : 0.6375
Precision : 0.6397
Recall : 0.6375
F1 Score : 0.6099


# 8. Multinomial Naïve Bayes

## Objective

Multinomial Naïve Bayes is a probabilistic classifier commonly used for text classification. It assumes conditional independence among features and provides a strong baseline for comparison with discriminative models.

In [ ]:
###############################################################
# MULTINOMIAL NAIVE BAYES
###############################################################

nb = MultinomialNB()

nb.fit(X_train_tfidf, y_train)

nb_predictions = nb.predict(X_test_tfidf)

print("Naive Bayes Training Completed!")

Naive Bayes Training Completed!


In [ ]:
###############################################################
# NAIVE BAYES RESULTS
###############################################################

nb_accuracy = accuracy_score(y_test, nb_predictions)

nb_precision = precision_score(

    y_test,

    nb_predictions,

    average="weighted"

)

nb_recall = recall_score(

    y_test,

    nb_predictions,

    average="weighted"

)

nb_f1 = f1_score(

    y_test,

    nb_predictions,

    average="weighted"

)

print(f"Accuracy : {nb_accuracy:.4f}")

print(f"Precision : {nb_precision:.4f}")

print(f"Recall : {nb_recall:.4f}")

print(f"F1 Score : {nb_f1:.4f}")

Accuracy : 0.3729
Precision : 0.3830
Recall : 0.3729
F1 Score : 0.2843


# 9. Linear Support Vector Machine

## Objective

Linear Support Vector Machine (LinearSVC) is implemented as the primary traditional machine learning classifier. Linear SVM is well suited for high-dimensional sparse text data represented using TF-IDF features and is widely regarded as one of the strongest baseline classifiers for text classification tasks.

In [ ]:
###############################################################
# LINEAR SUPPORT VECTOR MACHINE
###############################################################

svm = LinearSVC(

    random_state=42

)

svm.fit(X_train_tfidf, y_train)

svm_predictions = svm.predict(X_test_tfidf)

print("Linear SVM Training Completed!")

Linear SVM Training Completed!


In [ ]:
###############################################################
# SVM RESULTS
###############################################################

svm_accuracy = accuracy_score(y_test, svm_predictions)

svm_precision = precision_score(

    y_test,

    svm_predictions,

    average="weighted"

)

svm_recall = recall_score(

    y_test,

    svm_predictions,

    average="weighted"

)

svm_f1 = f1_score(

    y_test,

    svm_predictions,

    average="weighted"

)

print(f"Accuracy : {svm_accuracy:.4f}")

print(f"Precision : {svm_precision:.4f}")

print(f"Recall : {svm_recall:.4f}")

print(f"F1 Score : {svm_f1:.4f}")

Accuracy : 0.6438
Precision : 0.6456
Recall : 0.6438
F1 Score : 0.6359


# 10. Optimized TF-IDF Feature Extraction

## Objective

To improve the discriminative power of textual features, an optimized TF-IDF representation is constructed. The vectorizer is configured using a larger feature space, sublinear term frequency scaling, and bi-gram representations to capture richer contextual information for personality prediction.

In [ ]:
###############################################################
# OPTIMIZED TF-IDF
###############################################################

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_optimized = TfidfVectorizer(

    max_features=20000,

    ngram_range=(1,2),

    min_df=2,

    max_df=0.95,

    sublinear_tf=True

)

X_train_opt = tfidf_optimized.fit_transform(X_train)

X_test_opt = tfidf_optimized.transform(X_test)

print("Optimized TF-IDF Completed!")

print(X_train_opt.shape)
print(X_test_opt.shape)

Optimized TF-IDF Completed!
(6939, 20000)
(1735, 20000)


# 11. Feature Selection using Chi-Square

## Objective

Feature selection is applied to retain only the most informative textual features. Removing less relevant features reduces noise, improves computational efficiency, and may enhance classification performance.

In [ ]:
###############################################################
# FEATURE SELECTION
###############################################################

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

selector = SelectKBest(

    score_func=chi2,

    k=8000

)

X_train_selected = selector.fit_transform(

    X_train_opt,

    y_train

)

X_test_selected = selector.transform(

    X_test_opt

)

print("Feature Selection Completed!")

print(X_train_selected.shape)

Feature Selection Completed!
(6939, 8000)


# 12. Optimized Linear Support Vector Machine

## Objective

An optimized Linear Support Vector Machine is trained using the selected TF-IDF features. Class balancing is incorporated to improve learning across all personality categories.

In [ ]:
###############################################################
# OPTIMIZED LINEAR SVM
###############################################################

optimized_svm = LinearSVC(

    C=2,

    class_weight="balanced",

    random_state=42,

    max_iter=5000

)

optimized_svm.fit(

    X_train_selected,

    y_train

)

optimized_predictions = optimized_svm.predict(

    X_test_selected

)

print("Optimized SVM Trained Successfully!")

Optimized SVM Trained Successfully!


In [ ]:
###############################################################
# OPTIMIZED SVM RESULTS
###############################################################

optimized_accuracy = accuracy_score(

    y_test,

    optimized_predictions

)

optimized_precision = precision_score(

    y_test,

    optimized_predictions,

    average="weighted"

)

optimized_recall = recall_score(

    y_test,

    optimized_predictions,

    average="weighted"

)

optimized_f1 = f1_score(

    y_test,

    optimized_predictions,

    average="weighted"

)

print(f"Accuracy : {optimized_accuracy:.4f}")

print(f"Precision : {optimized_precision:.4f}")

print(f"Recall : {optimized_recall:.4f}")

print(f"F1 Score : {optimized_f1:.4f}")

Accuracy : 0.6870
Precision : 0.6851
Recall : 0.6870
F1 Score : 0.6835


In [ ]:
###############################################################
# MODEL COMPARISON
###############################################################

comparison = pd.DataFrame({

    "Model":[

        "Logistic Regression",

        "Naive Bayes",

        "Linear SVM",

        "Optimized Linear SVM"

    ],

    "Accuracy":[

        lr_accuracy,

        nb_accuracy,

        svm_accuracy,

        optimized_accuracy

    ],

    "Precision":[

        lr_precision,

        nb_precision,

        svm_precision,

        optimized_precision

    ],

    "Recall":[

        lr_recall,

        nb_recall,

        svm_recall,

        optimized_recall

    ],

    "F1 Score":[

        lr_f1,

        nb_f1,

        svm_f1,

        optimized_f1

    ]

})

comparison = comparison.sort_values(

    by="Accuracy",

    ascending=False

)

comparison

,Model,Accuracy,Precision,Recall,F1 Score
3,Optimized Linear SVM,0.687032,0.685125,0.687032,0.683544
2,Linear SVM,0.643804,0.645594,0.643804,0.635932
0,Logistic Regression,0.637464,0.639694,0.637464,0.609945
1,Naive Bayes,0.372911,0.382996,0.372911,0.284346


####Repating the same process for MBTI 500 dataset

In [ ]:
mbti500 = pd.read_csv(DATASET_PATH + "mbti500_preprocessed.csv")

In [ ]:
###############################################################
# DATA VALIDATION
###############################################################

print("Dataset Shape:", mbti500.shape)

print("\nColumns:")
print(mbti500.columns)

print("\nMissing Values:")
print(mbti500.isnull().sum())

print("\nDuplicate Rows:", mbti500.duplicated().sum())

mbti500.head()

Dataset Shape: (106067, 2)

Columns:
Index(['type', 'clean_text'], dtype='object')

Missing Values:
type          0
clean_text    0
dtype: int64

Duplicate Rows: 0


,type,clean_text
0,INTJ,know intj tool use interaction people excuse a...
1,INTJ,rap music ehh opp yeah know valid well know fa...
2,INTJ,preferably p hd low except wew lad video p min...
3,INTJ,drink like wish could drink red wine give head...
4,INTJ,space program ah bad deal meing freelance max ...


In [ ]:
###############################################################
# LABEL ENCODING
###############################################################

from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

mbti500["label"] = encoder.fit_transform(mbti500["type"])

print("Number of Classes:", mbti500["label"].nunique())

Number of Classes: 16


In [ ]:
###############################################################
# TRAIN TEST SPLIT
###############################################################

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    mbti500["clean_text"],
    mbti500["label"],
    test_size=0.20,
    random_state=42,
    stratify=mbti500["label"]
)

print("Training Samples :", len(X_train))
print("Testing Samples  :", len(X_test))

Training Samples : 84853
Testing Samples  : 21214


In [ ]:
###############################################################
# TF-IDF
###############################################################

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)

(84853, 20000)


In [ ]:
###############################################################
# FEATURE SELECTION
###############################################################

from sklearn.feature_selection import SelectKBest, chi2

selector = SelectKBest(score_func=chi2, k=8000)

X_train_selected = selector.fit_transform(X_train_tfidf, y_train)
X_test_selected = selector.transform(X_test_tfidf)

In [ ]:
###############################################################
# LOGISTIC REGRESSION
###############################################################

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr.fit(X_train_selected, y_train)

pred_lr = lr.predict(X_test_selected)

lr_accuracy = accuracy_score(y_test, pred_lr)
lr_precision = precision_score(y_test, pred_lr, average='weighted')
lr_recall = recall_score(y_test, pred_lr, average='weighted')
lr_f1 = f1_score(y_test, pred_lr, average='weighted')

print(classification_report(y_test, pred_lr))

print("\nAccuracy :", lr_accuracy)
print("Precision:", lr_precision)
print("Recall   :", lr_recall)
print("F1 Score :", lr_f1)

              precision    recall  f1-score   support

           0       0.86      0.40      0.55       307
           1       0.84      0.79      0.81      1233
           2       0.93      0.68      0.78       591
           3       0.85      0.82      0.84      2345
           4       1.00      0.03      0.05        36
           5       1.00      0.07      0.13        72
           6       0.97      0.62      0.76        96
           7       0.99      0.84      0.91       397
           8       0.82      0.87      0.84      2993
           9       0.80      0.84      0.82      2427
          10       0.82      0.89      0.85      4486
          11       0.83      0.90      0.86      4992
          12       0.87      0.20      0.33       130
          13       0.74      0.27      0.40       175
          14       0.88      0.36      0.51       249
          15       0.90      0.73      0.81       685

    accuracy                           0.83     21214
   macro avg       0.88   

In [ ]:
###############################################################
# NAIVE BAYES
###############################################################

from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()

nb.fit(X_train_selected, y_train)

pred_nb = nb.predict(X_test_selected)

nb_accuracy = accuracy_score(y_test, pred_nb)
nb_precision = precision_score(y_test, pred_nb, average='weighted')
nb_recall = recall_score(y_test, pred_nb, average='weighted')
nb_f1 = f1_score(y_test, pred_nb, average='weighted')

print(classification_report(y_test, pred_nb))

print("\nAccuracy :", nb_accuracy)
print("Precision:", nb_precision)
print("Recall   :", nb_recall)
print("F1 Score :", nb_f1)

              precision    recall  f1-score   support

           0       1.00      0.02      0.03       307
           1       0.94      0.04      0.08      1233
           2       1.00      0.06      0.11       591
           3       0.76      0.28      0.40      2345
           4       0.00      0.00      0.00        36
           5       0.00      0.00      0.00        72
           6       0.98      0.46      0.62        96
           7       0.94      0.59      0.73       397
           8       0.60      0.74      0.66      2993
           9       0.66      0.51      0.58      2427
          10       0.63      0.79      0.70      4486
          11       0.53      0.92      0.67      4992
          12       0.00      0.00      0.00       130
          13       0.00      0.00      0.00       175
          14       0.00      0.00      0.00       249
          15       0.98      0.06      0.12       685

    accuracy                           0.60     21214
   macro avg       0.56   

In [ ]:
###############################################################
# LINEAR SVM
###############################################################

from sklearn.svm import LinearSVC

svm = LinearSVC(random_state=42)

svm.fit(X_train_selected, y_train)

pred_svm = svm.predict(X_test_selected)

svm_accuracy = accuracy_score(y_test, pred_svm)
svm_precision = precision_score(y_test, pred_svm, average='weighted')
svm_recall = recall_score(y_test, pred_svm, average='weighted')
svm_f1 = f1_score(y_test, pred_svm, average='weighted')

print(classification_report(y_test, pred_svm))

print("\nAccuracy :", svm_accuracy)
print("Precision:", svm_precision)
print("Recall   :", svm_recall)
print("F1 Score :", svm_f1)

              precision    recall  f1-score   support

           0       0.78      0.61      0.68       307
           1       0.83      0.81      0.82      1233
           2       0.90      0.77      0.83       591
           3       0.85      0.84      0.85      2345
           4       0.82      0.39      0.53        36
           5       0.77      0.46      0.57        72
           6       0.90      0.75      0.82        96
           7       0.95      0.92      0.93       397
           8       0.84      0.85      0.85      2993
           9       0.82      0.82      0.82      2427
          10       0.84      0.88      0.86      4486
          11       0.84      0.88      0.86      4992
          12       0.89      0.55      0.68       130
          13       0.70      0.43      0.53       175
          14       0.81      0.65      0.72       249
          15       0.87      0.81      0.84       685

    accuracy                           0.84     21214
   macro avg       0.84   

In [ ]:
###############################################################
# OPTIMIZED LINEAR SVM
###############################################################

optimized_svm = LinearSVC(
    C=2,
    class_weight='balanced',
    max_iter=5000,
    random_state=42
)

optimized_svm.fit(X_train_selected, y_train)

pred_opt = optimized_svm.predict(X_test_selected)

opt_accuracy = accuracy_score(y_test, pred_opt)
opt_precision = precision_score(y_test, pred_opt, average='weighted')
opt_recall = recall_score(y_test, pred_opt, average='weighted')
opt_f1 = f1_score(y_test, pred_opt, average='weighted')

print(classification_report(y_test, pred_opt))

print("\nAccuracy :", opt_accuracy)
print("Precision:", opt_precision)
print("Recall   :", opt_recall)
print("F1 Score :", opt_f1)

              precision    recall  f1-score   support

           0       0.67      0.73      0.70       307
           1       0.77      0.84      0.80      1233
           2       0.79      0.84      0.81       591
           3       0.84      0.85      0.84      2345
           4       0.75      0.67      0.71        36
           5       0.63      0.56      0.59        72
           6       0.80      0.82      0.81        96
           7       0.90      0.92      0.91       397
           8       0.85      0.84      0.85      2993
           9       0.81      0.82      0.81      2427
          10       0.87      0.85      0.86      4486
          11       0.88      0.85      0.87      4992
          12       0.68      0.66      0.67       130
          13       0.59      0.59      0.59       175
          14       0.65      0.76      0.70       249
          15       0.79      0.87      0.83       685

    accuracy                           0.84     21214
   macro avg       0.77   

In [ ]:
###############################################################
# RESULTS SUMMARY
###############################################################

import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Naive Bayes",
        "Linear SVM",
        "Optimized Linear SVM"
    ],
    "Accuracy": [
        lr_accuracy,
        nb_accuracy,
        svm_accuracy,
        opt_accuracy
    ],
    "Precision": [
        lr_precision,
        nb_precision,
        svm_precision,
        opt_precision
    ],
    "Recall": [
        lr_recall,
        nb_recall,
        svm_recall,
        opt_recall
    ],
    "F1 Score": [
        lr_f1,
        nb_f1,
        svm_f1,
        opt_f1
    ]
})

results = results.sort_values(by="Accuracy", ascending=False)

results

,Model,Accuracy,Precision,Recall,F1 Score
2,Linear SVM,0.841048,0.840732,0.841048,0.839453
3,Optimized Linear SVM,0.838173,0.839695,0.838173,0.838600
0,Logistic Regression,0.830301,0.834015,0.830301,0.823285
1,Naive Bayes,0.597813,0.652754,0.597813,0.539389


### Combining the dataset


In [ ]:
###############################################################
# COMBINE MBTI1 AND MBTI500
###############################################################

combined_df = pd.concat([mbti1, mbti500], ignore_index=True)

print("Combined Dataset Shape:", combined_df.shape)

combined_df.head()

Combined Dataset Shape: (114742, 3)


,type,clean_text,label
0,INFJ,intj moment sportscenter top ten play prank li...,NaN
1,ENTP,I find lack I post alarm sex boring position o...,NaN
2,INTP,good one course I say I know my blessing my cu...,NaN
3,INTJ,dear intp I enjoy our conversation day esoteri...,NaN
4,ENTJ,you fire another silly misconception approach ...,NaN


In [ ]:
###############################################################
# REMOVE MISSING VALUES
###############################################################

combined_df = combined_df.dropna(subset=["clean_text"])

combined_df = combined_df[
    combined_df["clean_text"].str.strip() != ""
]

print("Shape after removing missing values:")
print(combined_df.shape)

Shape after removing missing values:
(114741, 3)


In [ ]:
###############################################################
# REMOVE DUPLICATE POSTS
###############################################################

before = combined_df.shape[0]

combined_df = combined_df.drop_duplicates(
    subset=["clean_text"]
)

after = combined_df.shape[0]

print("Rows before :", before)
print("Rows after  :", after)
print("Duplicates Removed :", before - after)

Rows before : 114741
Rows after  : 114741
Duplicates Removed : 0


In [ ]:
###############################################################
# SHUFFLE DATASET
###############################################################

combined_df = combined_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

combined_df.head()

,type,clean_text,label
0,ENTP,possibly ironic word mug also mug existence li...,3.0
1,INFP,aww man you poor poor guy I say I infp friend ...,NaN
2,ENTP,offer date marion heroic maid marion prioritiz...,3.0
3,INTP,resus may vary simple way expose large branch ...,11.0
4,INTJ,anything interest context whether contrast soc...,10.0


In [ ]:
###############################################################
# CLASS DISTRIBUTION
###############################################################

print(combined_df["type"].value_counts())

print("\nNumber of Classes:",
      combined_df["type"].nunique())

type
INTP    26265
INTJ    23518
INFJ    16433
INFP    13965
ENTP    12410
ENFP     6842
ISTP     3761
ENTJ     3186
ESTP     2075
ENFJ     1724
ISTJ     1448
ISFP     1146
ISFJ      816
ESTJ      521
ESFP      408
ESFJ      223
Name: count, dtype: int64

Number of Classes: 16


In [ ]:
###############################################################
# SAVE COMBINED DATASET
###############################################################

combined_df.to_csv(
    DATASET_PATH + "combined_mbti_preprocessed.csv",
    index=False
)

print("✅ Combined dataset saved successfully!")

✅ Combined dataset saved successfully!


In [ ]:
combined_df = pd.read_csv(
    DATASET_PATH + "combined_mbti_preprocessed.csv"
)

In [ ]:
###############################################################
# LOAD COMBINED DATASET
###############################################################

combined_df = pd.read_csv(
    DATASET_PATH + "combined_mbti_preprocessed.csv"
)

print("Dataset Shape:", combined_df.shape)
combined_df.head()

Dataset Shape: (114741, 3)


,type,clean_text,label
0,ENTP,possibly ironic word mug also mug existence li...,3.0
1,INFP,aww man you poor poor guy I say I infp friend ...,NaN
2,ENTP,offer date marion heroic maid marion prioritiz...,3.0
3,INTP,resus may vary simple way expose large branch ...,11.0
4,INTJ,anything interest context whether contrast soc...,10.0


In [ ]:
###############################################################
# DATA VALIDATION
###############################################################

print("Dataset Shape:", combined_df.shape)

print("\nColumns:")
print(combined_df.columns)

print("\nMissing Values:")
print(combined_df.isnull().sum())

print("\nDuplicate Rows:")
print(combined_df.duplicated().sum())

Dataset Shape: (114741, 3)

Columns:
Index(['type', 'clean_text', 'label'], dtype='object')

Missing Values:
type             0
clean_text       0
label         8674
dtype: int64

Duplicate Rows:
0


In [ ]:
###############################################################
# LABEL ENCODING
###############################################################

from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

combined_df["label"] = encoder.fit_transform(combined_df["type"])

print("Number of Classes:", combined_df["label"].nunique())

Number of Classes: 16


In [ ]:
###############################################################
# TRAIN TEST SPLIT
###############################################################

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    combined_df["clean_text"],
    combined_df["label"],
    test_size=0.20,
    random_state=42,
    stratify=combined_df["label"]
)

print("Training Samples :", len(X_train))
print("Testing Samples  :", len(X_test))

Training Samples : 91792
Testing Samples  : 22949


In [ ]:
###############################################################
# TF-IDF VECTORIZATION
###############################################################

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(91792, 20000)
(22949, 20000)


In [ ]:
###############################################################
# FEATURE SELECTION
###############################################################

from sklearn.feature_selection import SelectKBest, chi2

selector = SelectKBest(
    score_func=chi2,
    k=8000
)

X_train_selected = selector.fit_transform(
    X_train_tfidf,
    y_train
)

X_test_selected = selector.transform(
    X_test_tfidf
)

print(X_train_selected.shape)
print(X_test_selected.shape)

(91792, 8000)
(22949, 8000)


In [ ]:
###############################################################
# LOGISTIC REGRESSION
###############################################################

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr.fit(X_train_selected, y_train)

pred_lr = lr.predict(X_test_selected)

lr_accuracy = accuracy_score(y_test, pred_lr)
lr_precision = precision_score(y_test, pred_lr, average='weighted')
lr_recall = recall_score(y_test, pred_lr, average='weighted')
lr_f1 = f1_score(y_test, pred_lr, average='weighted')

In [ ]:
###############################################################
# NAIVE BAYES
###############################################################

from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()

nb.fit(X_train_selected, y_train)

pred_nb = nb.predict(X_test_selected)

nb_accuracy = accuracy_score(y_test, pred_nb)
nb_precision = precision_score(y_test, pred_nb, average='weighted')
nb_recall = recall_score(y_test, pred_nb, average='weighted')
nb_f1 = f1_score(y_test, pred_nb, average='weighted')

In [ ]:
###############################################################
# LINEAR SVM
###############################################################

from sklearn.svm import LinearSVC

svm = LinearSVC(random_state=42)

svm.fit(X_train_selected, y_train)

pred_svm = svm.predict(X_test_selected)

svm_accuracy = accuracy_score(y_test, pred_svm)
svm_precision = precision_score(y_test, pred_svm, average='weighted')
svm_recall = recall_score(y_test, pred_svm, average='weighted')
svm_f1 = f1_score(y_test, pred_svm, average='weighted')

In [ ]:
###############################################################
# OPTIMIZED LINEAR SVM
###############################################################

optimized_svm = LinearSVC(
    C=2,
    class_weight="balanced",
    max_iter=5000,
    random_state=42
)

optimized_svm.fit(X_train_selected, y_train)

pred_opt = optimized_svm.predict(X_test_selected)

opt_accuracy = accuracy_score(y_test, pred_opt)
opt_precision = precision_score(y_test, pred_opt, average='weighted')
opt_recall = recall_score(y_test, pred_opt, average='weighted')
opt_f1 = f1_score(y_test, pred_opt, average='weighted')

In [ ]:
###############################################################
# RESULTS SUMMARY
###############################################################

import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Naive Bayes",
        "Linear SVM",
        "Optimized Linear SVM"
    ],
    "Accuracy": [
        lr_accuracy,
        nb_accuracy,
        svm_accuracy,
        opt_accuracy
    ],
    "Precision": [
        lr_precision,
        nb_precision,
        svm_precision,
        opt_precision
    ],
    "Recall": [
        lr_recall,
        nb_recall,
        svm_recall,
        opt_recall
    ],
    "F1 Score": [
        lr_f1,
        nb_f1,
        svm_f1,
        opt_f1
    ]
})

results = results.sort_values(by="Accuracy", ascending=False)

results

,Model,Accuracy,Precision,Recall,F1 Score
2,Linear SVM,0.844612,0.845057,0.844612,0.843593
3,Optimized Linear SVM,0.841387,0.843648,0.841387,0.842037
0,Logistic Regression,0.826833,0.830521,0.826833,0.820747
1,Naive Bayes,0.564992,0.617164,0.564992,0.508208
